# LLM Few-Shot — Topic-Level Sentiment Results

Computes sentiment distribution per topic from `llm_topic_predictions.csv`.

- Input: `llm_topic_predictions.csv` — 99,834 reviews, each with its LDA-assigned topic and LLM sentiment prediction
- No ground truth exists for topic-level sentiment → distribution only, no F1
- N/A predictions (model failed to output a valid label) are dropped before aggregation

In [1]:
import pandas as pd

In [2]:
# Load topic-level predictions
df = pd.read_csv("llm_topic_predictions.csv")
print(f"Total rows: {len(df)}")
print(f"\npred_sentiment distribution:\n{df['pred_sentiment'].value_counts(dropna=False)}")

Total rows: 99834

pred_sentiment distribution:
pred_sentiment
positive    48819
negative    27342
NaN         23673
Name: count, dtype: int64


In [3]:
# Drop N/A predictions
df_valid = df[df["pred_sentiment"].isin(["positive", "negative"])].copy()
n_na = len(df) - len(df_valid)
print(f"N/A predictions dropped: {n_na} ({n_na / len(df) * 100:.1f}%)")
print(f"Valid predictions: {len(df_valid)}")

N/A predictions dropped: 23673 (23.7%)
Valid predictions: 76161


In [4]:
# Compute per-topic sentiment distribution
summary = (
    df_valid
    .groupby("topic_label")["pred_sentiment"]
    .value_counts()
    .unstack(fill_value=0)
    .rename(columns={"positive": "n_pos", "negative": "n_neg"})
)

for col in ["n_pos", "n_neg"]:
    if col not in summary.columns:
        summary[col] = 0

summary["total"]   = summary["n_pos"] + summary["n_neg"]
summary["pct_pos"] = (summary["n_pos"] / summary["total"] * 100).round(1)
summary["pct_neg"] = (summary["n_neg"] / summary["total"] * 100).round(1)

# Sort from most negative to most positive (matches report structure)
summary = summary.sort_values("pct_pos")

summary[["total", "n_pos", "n_neg", "pct_pos", "pct_neg"]]

pred_sentiment,total,n_pos,n_neg,pct_pos,pct_neg
topic_label,,,,,
Shipping damage,4497,784,3713,17.4,82.6
String quality,4820,1328,3492,27.6,72.4
Tuning stability,5053,1678,3375,33.2,66.8
Fret / neck setup,10203,4439,5764,43.5,56.5
Customer service / returns,5466,2584,2882,47.3,52.7
Electronics / controls,1438,903,535,62.8,37.2
Accessories,7147,4835,2312,67.7,32.3
Visual appearance,3342,2303,1039,68.9,31.1
Setup / action,3844,3072,772,79.9,20.1


In [5]:
# Pretty-print the results
print(f"{'Topic':<30} {'Total':>7} {'n_pos':>7} {'n_neg':>7} {'Pos%':>7} {'Neg%':>7}")
print("-" * 66)

for topic, row in summary.iterrows():
    print(
        f"{topic:<30} {int(row['total']):>7,}"
        f" {int(row['n_pos']):>7,}"
        f" {int(row['n_neg']):>7,}"
        f" {row['pct_pos']:>6.1f}%"
        f" {row['pct_neg']:>6.1f}%"
    )

Topic                            Total   n_pos   n_neg    Pos%    Neg%
------------------------------------------------------------------
Shipping damage                  4,497     784   3,713   17.4%   82.6%
String quality                   4,820   1,328   3,492   27.6%   72.4%
Tuning stability                 5,053   1,678   3,375   33.2%   66.8%
Fret / neck setup               10,203   4,439   5,764   43.5%   56.5%
Customer service / returns       5,466   2,584   2,882   47.3%   52.7%
Electronics / controls           1,438     903     535   62.8%   37.2%
Accessories                      7,147   4,835   2,312   67.7%   32.3%
Visual appearance                3,342   2,303   1,039   68.9%   31.1%
Setup / action                   3,844   3,072     772   79.9%   20.1%
Playability / chords             4,437   3,612     825   81.4%   18.6%
Guitar size                      4,594   3,767     827   82.0%   18.0%
Pickups                          5,162   4,584     578   88.8%   11.2%
Beginner l

In [6]:
# Save to CSV (overwrites llm_topic_sentiment.csv with recomputed values)
output = summary[["total", "n_pos", "n_neg", "pct_pos", "pct_neg"]].copy()
output.index.name = "topic"
output.to_csv("llm_topic_sentiment.csv")
print("Saved to llm_topic_sentiment.csv")

Saved to llm_topic_sentiment.csv
